# NB 2.3 &mdash; Regressió polinòmica: quan el model aprèn massa

**MP 5134** &mdash; UT2 &mdash; *Versió demanda elèctrica*

*Dades: demanda elèctrica diària de les Illes Balears (Red Eléctrica) i el temps de cada dia a l'aeroport de Palma (AEMET).*

---
### Què farem avui

Al final del NB 2.2 vam topar amb un límit. La latitud i el preu dels habitatges
dibuixaven dues muntanyes, i el nostre model només sabia traçar rectes. Avui
aprendrem a **doblegar la recta**.

Per aprendre-ho canviarem de dades: deixem Califòrnia i tornem a casa. Predirem
**quanta electricitat gasten les Balears en un dia** a partir del temps que fa. És
un problema real, perquè Red Eléctrica ha de saber cada dia quanta energia ha de
tenir a punt, i té una forma que una recta no pot seguir de cap manera.

Però aquest notebook no va realment de corbes. Va del problema més important de
tot l'aprenentatge automàtic, i el veurem amb els nostres ulls: **un model pot
aprendre tan bé les dades d'entrenament que deixi de servir per a dades noves.**

Quan acabis hauries de saber respondre aquestes preguntes:

- Com pot una regressió *lineal* dibuixar una corba?
- Què és el sobreajust i què és l'infraajust?
- Com es detecta el sobreajust, si no el podem dibuixar?
- Què podem fer quan un model sobreajusta?
- I tot això, serveix quan el model té moltes variables?
- Per què el test segellat no es pot mirar fins al final del curs?

Pel camí trobaràs tres pauses **Ara tu**, com al NB 2.1. Són exercicis curts,
lligats a la secció que acabes de veure, perquè provis amb les teves mans el que
s'acaba d'explicar abans de continuar.

## 1. Les dades, amb el segell posat

Avui estrenem fitxer. Cada fila és un dia, del gener de 2015 al desembre de 2025,
i ajunta dues fonts: la **demanda elèctrica de les Balears**, que publica Red
Eléctrica, l'empresa que gestiona la xarxa, i **el temps que va fer aquell dia a
l'aeroport de Palma**, mesurat per l'AEMET, l'agència estatal de meteorologia.

Abans de mirar res, fem el mateix que al NB 2.2: apartem un tros de les dades i el
deixem **sota clau fins a la UT10**. Allà vam apartar un 20% dels districtes a
l'atzar. Aquí tallarem **per dates**: el 2024 i el 2025 queden segellats. Volem un
model que serveixi per als dies que han de venir, i la manera honesta de
comprovar-ho és examinar-lo amb anys que encara no ha vist. A més, dos dies seguits
s'assemblen tant que, si els triéssim a l'atzar, el model s'examinaria de dies que
són gairebé còpies dels que ha estudiat.

Amb el que queda fem com al NB 2.2: un tros per entrenar, del 2015 al 2021, i un
per validar, el 2022 i el 2023.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL_DADES = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT02-Regressio_lineal_polinomica/demanda_electrica/demanda_balears.csv"
df = pd.read_csv(URL_DADES, parse_dates=["fecha"])

# El test segellat (2024-2025) no es toca fins a la UT10
TALL_SEGELLAT = "2024-01-01"
dades = df[df["fecha"] < TALL_SEGELLAT].copy()

TALL_VALIDACIO = "2022-01-01"
train = dades[dades["fecha"] < TALL_VALIDACIO].copy()
valid = dades[dades["fecha"] >= TALL_VALIDACIO].copy()

print(f"Dies d'entrenament (2015-2021): {len(train)}")
print(f"Dies de validació  (2022-2023): {len(valid)}")
dades.head()

Mirem què hi ha a cada columna:

- `demanda`, l'electricitat que han gastat les quatre illes aquell dia, en
  **gigawatts hora** (GWh). Un GWh és un milió de kWh, la unitat que surt a la
  factura de la llum de casa. Un dia normal, les Balears en gasten entre 13 i
  18 GWh. És el que volem predir.
- `tmed`, `tmax` i `tmin`, la temperatura mitjana, la màxima i la mínima del dia,
  en °C.
- `prec`, la pluja, en litres per metre quadrat; `sol`, les hores de sol; i
  `velmedia`, la velocitat mitjana del vent, en metres per segon.
- `cap_de_setmana`, que val 1 els dissabtes i els diumenges i 0 la resta de dies.
- `any`, `mes` i `dia_any`, el calendari. `dia_any` és el número del dia dins de
  l'any: l'1 de gener és el dia 1, i el 31 de desembre, el 365 (o el 366 els anys
  de traspàs).

Fins a la secció 8 només en farem servir dues. Volem predir la `demanda` d'un dia a
partir d'una sola dada, la **temperatura mitjana**. Amb una sola entrada, tot el
que faci el model es pot dibuixar, i això té una virtut enorme per aprendre.

## 2. L'electricitat segons la temperatura

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(train["tmed"], train["demanda"], alpha=0.15, s=8)
plt.xlabel("Temperatura mitjana (°C)")
plt.ylabel("Demanda elèctrica (GWh)")
plt.title("Set anys de consum elèctric, dia a dia")
plt.show()

Cada punt és un dia de 2015 a 2021, i es veu una forma molt clara: una **U**, una
mica torta. Els dies que gasten menys són els d'entre 13 i 16 °C, amb uns
13,5 GWh. Quan fa fred, el consum puja, perquè s'encenen les calefaccions. I quan
fa calor puja molt més, per l'aire condicionat i perquè a l'estiu a les illes hi
ha molta més gent: els dies de més de 28 °C passen dels 23 GWh.

Al voltant de la U hi ha gruix, perquè no tots els dies de 25 °C gasten el mateix:
un diumenge gasta menys que un dimarts. Fixa't també en una franja de punts
clarament més baixos que la resta entre els 16 i els 24 °C, i en uns quants més a
la banda de la calor. Són sobretot dies del 2020: la primavera del confinament i
un estiu amb molts menys turistes. No els traurem, perquè són dades reals, però en
parlarem al debat del final.

## 3. Una recta no pot fer-ho

Abans de res, preparem una columna nova: **la temperatura escalada**, que és
simplement la temperatura dividida entre 30. És el mateix número, però en una
escala més petita: el dia més calorós dels set anys val poc més d'1. De seguida
veuràs per què ens convé.

In [ ]:
for taula in (train, valid):
    taula["temp_escalada"] = taula["tmed"] / 30

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

recta = LinearRegression().fit(train[["temp_escalada"]], train["demanda"])

totes_les_temperatures = pd.DataFrame({"temp_escalada": np.linspace(0.1, 1.1, 200)})
temperatures_grafica = totes_les_temperatures["temp_escalada"] * 30

plt.figure(figsize=(8, 4))
plt.scatter(train["tmed"], train["demanda"], alpha=0.15, s=8)
plt.plot(temperatures_grafica, recta.predict(totes_les_temperatures), color="tab:red", linewidth=2)
plt.xlabel("Temperatura mitjana (°C)")
plt.ylabel("Demanda elèctrica (GWh)")
plt.show()

print(f"MAE de validació de la recta: {mean_absolute_error(valid['demanda'], recta.predict(valid[['temp_escalada']])):.2f} GWh")

Primer, com es llegeix aquest error. Quan diem que el model s'equivoca **2 GWh**
volem dir, per exemple, que prediu 17 GWh un dia en què se'n van gastar 15.
Tingues present que, si diguéssim sempre el consum mitjà dels anys
d'entrenament, ens equivocaríem uns 3 GWh.

La recta ho intenta, però no té cap manera de fer una U. Com que la calor fa pujar
el consum més que el fred, la recta s'inclina cap amunt i queda per sota dels
punts als dies freds i als més calorosos, i per sobre als del mig. S'equivoca uns
**2,1 GWh** de mitjana: ha recorregut només un terç del camí que hi ha entre dir
sempre la mitjana i no equivocar-se mai.

Aquest problema té nom. Quan un model és **massa simple per a la forma que tenen les
dades**, diem que fa **infraajust** (*underfitting*). L'infraajust es reconeix
perquè el model s'equivoca molt **fins i tot amb les dades d'entrenament**: no és
que no sàpiga generalitzar, és que no ha arribat a aprendre.

## 4. Com doblegar una recta

Aquí ve el truc, i és molt més senzill del que sembla.

Una regressió lineal **només sap sumar columnes multiplicades per un número**. No
la podem canviar. El que sí podem fer és **donar-li columnes noves**, fabricades a
partir de la que ja tenim: la temperatura escalada multiplicada per ella mateixa,
una altra vegada per ella mateixa, i així successivament.

Scikit-learn té una eina que fabrica aquestes columnes: `PolynomialFeatures`.
Mirem què fa amb tres dies d'exemple, de 9, 15 i 24 °C.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

exemple = pd.DataFrame({"temp_escalada": [0.3, 0.5, 0.8]})

fabrica = PolynomialFeatures(degree=3, include_bias=False)
columnes_noves = fabrica.fit_transform(exemple)

pd.DataFrame(columnes_noves, columns=fabrica.get_feature_names_out())

D'**una** columna n'hem fet **tres**:

- `temp_escalada`, la original.
- `temp_escalada^2`, la mateixa multiplicada per ella mateixa (0,5 × 0,5 = 0,25).
- `temp_escalada^3`, multiplicada tres vegades (0,5 × 0,5 × 0,5 = 0,125).

Amb aquestes tres columnes, el model continua fent el mateix de sempre: sumar-les
multiplicades per un número. **Però el resultat ja no és una recta**, perquè
aquestes columnes creixen de maneres diferents i, combinades, poden fer pujades i
baixades. No cal que sàpigues per què funciona; el que importa és veure-ho.

El número `degree` és el **grau**: fins a quantes multiplicacions arribem. Grau 1
és la recta de sempre. Grau 2 permet una corba amb una sola pujada o baixada.
Com més grau, més revolts pot fer la corba.

Ara ja pots entendre per què hem dividit la temperatura entre 30. Un dia de 33 °C
multiplicat quinze vegades per ell mateix dóna un número de vint-i-tres xifres, i
els ordinadors perden precisió amb números així. La temperatura escalada no passa
gaire d'1, i les seves potències es mantenen petites.

`include_bias=False` evita que la fàbrica afegeixi una columna plena d'uns. No la
necessitem, perquè `LinearRegression` ja té la seva ordenada a l'origen.

Recorda la taula de la UT1: **el grau és un hiperparàmetre**. El model no el tria;
el triem nosaltres. I tota la resta del notebook va de com triar-lo bé.

### 4.1 Corbes de grau 2 i de grau 4

Preparem tres funcions petites: una que fabrica les columnes i entrena el model,
una que fa prediccions i una que en calcula l'error. Les farem servir moltes
vegades.

In [ ]:
def entrenar_polinomi(grau, dades_entrenament):
    """Fabrica les columnes polinòmiques i entrena una regressió lineal amb elles."""
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X = fabrica.fit_transform(dades_entrenament[["temp_escalada"]])
    model = LinearRegression().fit(X, dades_entrenament["demanda"])
    return fabrica, model


def predir_polinomi(fabrica, model, taula):
    return model.predict(fabrica.transform(taula[["temp_escalada"]]))


def mae_polinomi(fabrica, model, taula):
    return mean_absolute_error(taula["demanda"], predir_polinomi(fabrica, model, taula))

Fixa't que, per entrenar, la fàbrica fa `fit_transform`, i per predir, només
`transform`. Totes les eines de scikit-learn que transformen dades funcionen així:
`fit` vol dir *prepara't mirant aquestes dades*, i `transform`, *converteix aquesta
taula amb el que has preparat*; `fit_transform` fa les dues coses de cop. La regla
és que la fàbrica es prepara **només amb les dades d'entrenament**, i després
aquella mateixa fàbrica converteix tot el que vingui: la validació, les temperatures
de la gràfica i, a la UT10, el test segellat. Per això `entrenar_polinomi` torna la
fàbrica juntament amb el model.

En el cas de `PolynomialFeatures`, el `fit` gairebé no aprèn res: només mira quantes
columnes li arriben i com es diuen, i 0,5 sempre donarà 0,25 i 0,125. Però a la UT3
veuràs eines que sí que aprenen números de les dades, com la mitjana de cada
columna, i allà preparar-les amb la validació seria fer trampa: el model faria
servir informació de dies que se suposa que encara no coneix. És el que s'anomena
*fuga d'informació*, i val més agafar el bon costum des del primer dia. (La
temperatura escalada no té aquest problema: la dividim sempre entre 30, un número
fix que no depèn de les dades.)

Ara sí, dibuixem les corbes de grau 2 i de grau 4.

In [ ]:
fig, grafics = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for grafic, grau in zip(grafics, [2, 4]):
    fabrica, model = entrenar_polinomi(grau, train)
    grafic.scatter(train["tmed"], train["demanda"], alpha=0.15, s=8)
    grafic.plot(temperatures_grafica, predir_polinomi(fabrica, model, totes_les_temperatures), color="tab:red", linewidth=2)
    grafic.set_title(f"Grau {grau}: MAE de validació {mae_polinomi(fabrica, model, valid):.2f} GWh")
    grafic.set_xlabel("Temperatura mitjana (°C)")

grafics[0].set_ylabel("Demanda elèctrica (GWh)")
plt.show()

El grau 2 ja fa la U i baixa l'error de 2,1 a 1,2 GWh: **gairebé la meitat**, i
sense cap dada nova, només amb una columna fabricada. El grau 4 hi afegeix poc:
segueix una mica millor el tram d'entre 20 i 28 °C, però l'error amb prou feines es
mou, i als extrems, on hi ha pocs dies, la corba ja fa coses estranyes: per sobre
dels 30 °C deixa de pujar. Aquesta forma és senzilla, una baixada i una pujada, i
per a això n'hi ha prou amb un sol revolt.

Fins aquí, tot són bones notícies: més grau, millor model, o com a mínim no
pitjor. Anem a veure on s'acaba la festa.

---
### Ara tu · Exercici 1: entrenament contra validació

Al NB 2.1 vas comparar l'error d'un model amb les dades d'entrenament i amb les de
prova. Fes el mateix aquí: amb `mae_polinomi`, calcula el MAE d'entrenament i el
de validació dels graus 1, 2 i 4 entrenats amb `train`, i posa'ls en una taula.

Els dos errors de cada grau s'assemblen o són molt diferents? I la recta,
s'equivoca molt només amb els dies nous o també amb els que ha vist? Guarda't la
resposta, perquè d'aquí a una estona la necessitaràs.

Pista: `entrenar_polinomi(1, train)` també funciona. Amb grau 1, la fàbrica deixa
la columna tal com és i surt la recta de sempre.

In [ ]:
# Escriu aquí el teu codi

## 5. Quan tenim poques dades

Fins ara hem entrenat amb set anys de dies, més de dues mil cinc-centes mostres.
Imagina ara que ens demanen el mateix model per a un sistema elèctric nou, del
qual només tenim **quinze dies de mesures**. És una situació molt més habitual
del que sembla: moltes vegades les dades són cares o escasses.

In [ ]:
poques = train.sample(15, random_state=2)

plt.figure(figsize=(8, 4))
plt.scatter(poques["tmed"], poques["demanda"], s=50, color="black")
plt.xlabel("Temperatura mitjana (°C)")
plt.ylabel("Demanda elèctrica (GWh)")
plt.title("Només quinze dies de mesures")
plt.xlim(3, 33)
plt.show()

Quinze punts. Es continua intuint la U, però hi ha forats grans: **no tenim cap
dia de fred de debò**, cap per sota dels 10 °C, ni cap dia d'entre 19 i 24 °C.

Entrenem-hi tres models: un de massa simple, un de raonable i un de molt complex.
Per a cadascun mirarem **dos errors**: el que fa amb aquests mateixos quinze dies i
el que fa amb els dos anys de validació.

In [ ]:
fig, grafics = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for grafic, grau in zip(grafics, [1, 2, 15]):
    fabrica, model = entrenar_polinomi(grau, poques)
    error_train = mae_polinomi(fabrica, model, poques)
    error_valid = mae_polinomi(fabrica, model, valid)

    grafic.scatter(valid["tmed"], valid["demanda"], alpha=0.1, s=6, color="gray", label="validació")
    grafic.scatter(poques["tmed"], poques["demanda"], s=40, color="black", label="els 15 dies")
    grafic.plot(temperatures_grafica, predir_polinomi(fabrica, model, totes_les_temperatures), color="tab:red", linewidth=2)
    grafic.set_ylim(5, 30)
    grafic.set_title(f"Grau {grau}\nentrenament {error_train:.2f} GWh · validació {error_valid:.1f} GWh")
    grafic.set_xlabel("Temperatura mitjana (°C)")

grafics[0].set_ylabel("Demanda elèctrica (GWh)")
grafics[0].legend(loc="upper center")
plt.show()

Aquestes tres gràfiques són el centre de tot el notebook. Mira-les amb calma.

**Grau 1.** La recta de sempre. S'equivoca molt amb els quinze dies i encara més
amb la validació. Infraajust.

**Grau 2.** La corba passa a prop dels punts sense tocar-los tots i dibuixa una U
creïble. S'equivoca 1,2 GWh amb els quinze dies i 1,5 amb la validació. És un model
raonable.

**Grau 15.** Aquí passa una cosa sorprenent. **La corba passa pràcticament per
sobre dels quinze punts**: l'error d'entrenament és de tres centèsimes de GWh,
gairebé perfecte. I en canvi, entre punt i punt fa **disbarats**: puja i baixa de
manera salvatge i surt del gràfic. L'error de validació passa de quaranta mil GWh,
més de dues mil vegades el que gasten les illes en un dia.

Hem hagut de retallar l'eix vertical perquè la gràfica es pogués llegir. Mira fins
on arriba de veritat:

In [ ]:
fabrica_15, model_15 = entrenar_polinomi(15, poques)
prediccions_15 = predir_polinomi(fabrica_15, model_15, totes_les_temperatures)

print(f"Demanda més baixa que prediu el grau 15: {prediccions_15.min():,.0f} GWh")
print(f"Demanda més alta que prediu el grau 15:  {prediccions_15.max():,.0f} GWh")

Un model que prediu un consum de **menys trenta-tres milions de GWh** els dies més
freds, i de **més de dos milions** els més calorosos, en una cosa que no pot ser
negativa i que no ha passat mai dels 30 GWh. Són justament els llocs on no hi
havia cap dels quinze punts per retenir la corba. I, alhora, és el model que
millor s'ajusta a les dades d'entrenament.

Això és el **sobreajust** (*overfitting*): el model té tanta llibertat que, en lloc
d'aprendre la forma general de les dades, **les memoritza**, soroll inclòs. Els
quinze punts tenen petites irregularitats de dies concrets que no es tornaran a
repetir, i el grau 15 ha doblegat la corba per passar exactament per totes elles.
Entre punt i punt, i sobretot més enllà de l'últim, no hi ha res que el retingui.

És com l'alumne que es prepara un examen memoritzant les respostes del de l'any
passat: ho clava si li posen les mateixes preguntes, i no sap fer res si canvien
una coma.

---
### Ara tu · Exercici 2: un dia de fred

Amb els quinze dies d'aquesta secció (`poques`), fes que el model de grau 2 i el
de grau 15 prediguin la demanda d'un dia de **7 °C** de mitjana. Compara-ho amb la
demanda mitjana real dels dies d'entrenament que van tenir entre 6 i 8 °C. Per què
el grau 15 falla tant precisament aquí? I el grau 2, encerta? Torna a mirar la
gràfica dels quinze punts.

Pista: per predir un sol dia, les funcions necessiten una taula amb la columna
`temp_escalada`, com ara `pd.DataFrame({"temp_escalada": [7 / 30]})`.

In [ ]:
# Escriu aquí el teu codi

## 6. Com es detecta el sobreajust sense dibuixar-lo

Amb una sola variable hem pogut dibuixar la corba i veure-hi els disbarats. Però
els models de veritat tenen deu, cent o mil variables, i no es poden dibuixar. Com
sabem, llavors, si un model sobreajusta?

La resposta és a les dues columnes que hem estat mirant: **l'error d'entrenament i
l'error de validació**. Fem-ho per a tots els graus de l'1 al 15.

In [ ]:
escombrada = []
for grau in range(1, 16):
    fabrica, model = entrenar_polinomi(grau, poques)
    escombrada.append({
        "grau": grau,
        "MAE entrenament": mae_polinomi(fabrica, model, poques),
        "MAE validació": mae_polinomi(fabrica, model, valid),
    })

escombrada = pd.DataFrame(escombrada)
escombrada.round(2)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(escombrada["grau"], escombrada["MAE entrenament"], marker="o", label="entrenament (els 15 dies)")
plt.plot(escombrada["grau"], escombrada["MAE validació"], marker="o", label="validació (2022-2023)")
plt.ylim(0, 5)
plt.xticks(range(1, 16))
plt.xlabel("Grau del polinomi")
plt.ylabel("MAE (GWh)")
plt.legend()
plt.title("L'error d'entrenament baixa; el de validació, no")
plt.show()

Aquesta gràfica, amb aquesta forma, la tornaràs a veure moltes vegades durant el
curs. Fixa-t'hi bé.

**La línia d'entrenament gairebé només baixa.** Cada grau més dóna més llibertat a
la corba per acostar-se als quinze punts, i per tant l'error d'entrenament
millora. Si només miréssim aquesta línia, triaríem el grau 15 i ens equivocaríem
de ple.

**La línia de validació fa una vall.** Al principi baixa, perquè el model passa de
massa simple a raonable. Arriba a un mínim, aquí al grau 2. I a partir d'aquí
torna a pujar, primer poc i després de manera desbocada: el model ha començat a
memoritzar. (Hem retallat l'eix a 5 GWh; els últims graus se'n van a milers.)

D'aquí surt la regla per detectar-ho:

| Què veus | Diagnòstic |
|---|---|
| Error alt a l'entrenament **i** a la validació | **Infraajust**: el model és massa simple |
| Error baix a l'entrenament **i** a la validació, semblants | Model raonable |
| Error baix a l'entrenament però **molt més alt** a la validació | **Sobreajust**: el model memoritza |

**El senyal d'alarma és la distància entre les dues línies.** Recorda el NB 2.2: el
model de vuit variables s'equivocava pràcticament igual amb els districtes que
havia vist i amb els que no. Ara ja saps per què això era una bona notícia.

I aquest és el motiu de fons pel qual existeix el conjunt de validació: **per
triar els hiperparàmetres**, com el grau, sense fer trampa amb el test. Però
triar també té un preu, i el veurem a la secció 9.

---
### Ara tu · Exercici 3: uns altres quinze dies

Repeteix l'escombrada d'aquesta secció triant uns altres quinze dies: canvia
`random_state=2` per `0`, `1` i `14`. El millor grau és sempre el 2? Què et diu
això sobre la confiança que podem tenir en una tria feta amb tan poques dades?

Pista: desa els dies nous en una variable amb un altre nom (per exemple,
`altres`), perquè `poques` la tornarem a necessitar a la secció 7.

In [ ]:
# Escriu aquí el teu codi

## 7. I si tinguéssim més dades?

Tornem al grau 15, el model desbocat. Què passa si, en lloc de quinze dies, li
donem els més de dos mil cinc-cents dies d'entrenament?

In [ ]:
fig, grafics = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for grafic, (nom, taula) in zip(grafics, [("15 dies", poques), ("tots els dies de 2015-2021", train)]):
    fabrica, model = entrenar_polinomi(15, taula)
    grafic.scatter(taula["tmed"], taula["demanda"], alpha=0.3 if len(taula) > 100 else 1,
                s=8 if len(taula) > 100 else 40, color="black")
    grafic.plot(temperatures_grafica, predir_polinomi(fabrica, model, totes_les_temperatures), color="tab:red", linewidth=2)
    grafic.set_ylim(5, 30)
    grafic.set_title(f"Grau 15 amb {nom}\nentrenament {mae_polinomi(fabrica, model, taula):.2f} GWh · "
                  f"validació {mae_polinomi(fabrica, model, valid):.2f} GWh")
    grafic.set_xlabel("Temperatura mitjana (°C)")

grafics[0].set_ylabel("Demanda elèctrica (GWh)")
plt.show()

**El mateix grau 15, amb moltes dades, es comporta bé.** La corba segueix la U i
l'error de validació torna a ser d'1,2 GWh, com el del grau 2 de la secció 4.1. Els
errors d'entrenament i de validació tornen a ser semblants. Només fa alguna ona als
extrems, per sota dels 7 °C i per sobre dels 29 °C, que és on hi ha menys dies: el
sobreajust sempre comença pels llocs amb poques dades.

Què ha passat? Amb milers de punts repartits per totes les temperatures, **no hi
ha forats on la corba es pugui desbocar**, i cap punt sol no la pot estirar cap a
ell perquè n'hi ha molts altres al voltant que la retenen.

La conclusió és important: **el sobreajust no depèn només del model, depèn de la
relació entre la complexitat del model i la quantitat de dades.** Un model massa
complex per a quinze dades pot ser perfectament raonable per a dues mil.

Quan un model sobreajusta, doncs, hi ha dues sortides:

- **Fer el model més simple** (aquí, baixar el grau).
- **Aconseguir més dades**, quan es pot.

Veuràs aquesta mateixa història amb altres models. A la **UT5**, el grau es dirà
*profunditat de l'arbre*. A la **UT6**, el paràmetre `C` de les SVM. A la **UT7**
aprendràs a triar aquests hiperparàmetres d'una manera molt més fiable que amb
una sola partició de validació.

## 8. I amb totes les variables?

Fins ara hem fet servir una sola entrada, la temperatura mitjana, perquè així es
podia dibuixar. Però el fitxer en té moltes més: la màxima i la mínima, la pluja,
les hores de sol, el vent, si és cap de setmana i en quin moment de l'any som. Al
NB 2.2 vam fer servir totes les variables d'una vegada amb una regressió lineal
múltiple. Fem el mateix aquí, i després hi apliquem les eines d'avui.

In [ ]:
from sklearn.metrics import r2_score

FEATURES = ["tmed", "tmax", "tmin", "prec", "sol", "velmedia", "cap_de_setmana", "dia_any"]
OBJECTIU = "demanda"

train_tot = train.dropna(subset=FEATURES)
valid_tot = valid.dropna(subset=FEATURES)


def provar(nom, X_train, X_valid):
    """Entrena una regressió lineal amb les columnes donades i en mesura els errors."""
    model = LinearRegression().fit(X_train, train_tot[OBJECTIU])
    return {
        "model": nom,
        "columnes": X_train.shape[1],
        "MAE entrenament": mean_absolute_error(train_tot[OBJECTIU], model.predict(X_train)),
        "MAE validació": mean_absolute_error(valid_tot[OBJECTIU], model.predict(X_valid)),
        "R2 validació": r2_score(valid_tot[OBJECTIU], model.predict(X_valid)),
    }


print(f"Dies d'entrenament complets: {len(train_tot)} de {len(train)}")
print(f"Dies de validació complets:  {len(valid_tot)} de {len(valid)}")

Hi ha uns pocs dies en què l'estació no va registrar les hores de sol o el vent, i
`fit()` no accepta valors absents: el `dropna()` els treu. Són nou dies de més de
tres mil, i no canvien res.

La funció `provar` ens estalviarà repetir codi: rep les columnes d'entrenament i
les de validació, entrena una regressió lineal i torna els errors, a més del
nombre de columnes que ha fet servir el model. Aquesta vegada hi afegim el R2 de
validació, com al NB 2.2. Al final de la secció veuràs per què el necessitem.

Comencem pel que ja saps fer: una regressió lineal amb les vuit variables. I després
hi afegim la idea d'avui, **la corba de la temperatura**: les columnes de grau 2 de
la temperatura escalada, les que han fet la U a la secció 4.1.

In [ ]:
corba = PolynomialFeatures(degree=2, include_bias=False)
corba_train = corba.fit_transform(train_tot[["temp_escalada"]])
corba_valid = corba.transform(valid_tot[["temp_escalada"]])

comparacio = pd.DataFrame([
    provar("tmed sola (secció 3)", train_tot[["tmed"]], valid_tot[["tmed"]]),
    provar("vuit variables", train_tot[FEATURES], valid_tot[FEATURES]),
    provar("vuit variables + corba de la temperatura",
           np.hstack([train_tot[FEATURES], corba_train]),
           np.hstack([valid_tot[FEATURES], corba_valid])),
])
comparacio.round(3)

`np.hstack` enganxa les columnes una al costat de l'altra: les vuit variables i les
dues de la corba, deu en total.

**Afegir variables a una recta gairebé no ajuda.** Amb vuit variables, l'error
baixa de 2,11 a 2,00 GWh, i el R2 passa de 0,53 a 0,58. És la mateixa lliçó del
NB 2.2: més variables només ajuden si porten informació que el model pugui fer
servir. Aquí la porten, però el model no la pot aprofitar, perquè totes entren en
línia recta. La màxima i la mínima, per exemple, també fan una U, i una recta
tampoc no la pot seguir.

**Doblegar la recta, en canvi, ho canvia tot.** Amb les mateixes vuit variables més
la corba de la temperatura, l'error baixa a 1,10 GWh i el R2 puja a 0,86. Dues
columnes fabricades valen molt més que set variables noves, perquè ara el model sap
que el consum puja pels dos costats de la U.

Fixa't també que, aquí, l'error de validació surt **més baix** que el
d'entrenament. No és cap error. A l'entrenament hi ha el 2020, amb uns mesos de
confinament que cap model del temps pot encertar, i la validació són dos anys
normals.

### 8.1 Combinar les variables entre elles

`PolynomialFeatures` encara pot fer una cosa més. Si li donem **diverses columnes**
en lloc d'una, no només fabrica els quadrats de cadascuna: també les **multiplica
entre elles**, de dues en dues. Amb grau 2 i les vuit variables surten 44 columnes:
les vuit originals, els vuit quadrats i els 28 productes de parelles, com
`tmax × cap_de_setmana` o `tmed × dia_any`. Aquests productes permeten al model
aprendre coses com que *la calor no fa gastar igual un diumenge que un dimarts*, o
que *25 °C al juny no són el mateix que 25 °C a l'agost, quan les illes són plenes
de gent*.

Fem l'escombrada de la secció 6, però ara amb totes les variables.

In [ ]:
escombrada_tot = []
for grau in (1, 2, 3, 4):
    fabrica = PolynomialFeatures(degree=grau, include_bias=False)
    X_train = fabrica.fit_transform(train_tot[FEATURES])
    X_valid = fabrica.transform(valid_tot[FEATURES])
    escombrada_tot.append(provar(f"vuit variables, grau {grau}", X_train, X_valid))

pd.DataFrame(escombrada_tot).round(3)

Quatre files, i hi ha tota la història del notebook:

- **Grau 1** són les vuit variables en línia recta: 2,00 GWh.
- **Grau 2**, amb 44 columnes, baixa l'error de validació a 1,03 GWh, millor fins i
  tot que la corba de la temperatura sola. Els productes entre variables aporten
  informació que una simple suma no podia veure.
- **Grau 3**, amb 164 columnes, encara millora una mica, fins a 0,95 GWh.
- **Grau 4**, amb 494 columnes, és el sobreajust de la secció 5, però ara amb el
  problema de veritat. L'error d'entrenament continua baixant, fins a 0,95 GWh,
  però el de validació **puja** fins a 1,44 GWh. I mira el R2: **−4,4**. Un R2
  negatiu vol dir que el model és pitjor que dir sempre el consum mitjà.

Per què el MAE puja tan poc i el R2 s'enfonsa tant? Perquè el grau 4 encerta la
majoria de dies, però en uns quants fa **disbarats enormes**, com el grau 15 de la
secció 5. Per a un dia de maig del 2022 amb 52 litres de pluja i, alhora, gairebé
onze hores de sol, una combinació que el model no havia vist mai, prediu un
consum de menys 220 GWh. El MAE és una
mitjana d'errors, i cinc dies dolents entre set-cents hi pesen poc. El R2 eleva
els errors al quadrat, i un sol error de 200 GWh hi pesa més que tots els altres
dies junts. **Per això convé mirar més d'una mètrica**: el MAE et diu com
s'equivoca el model un dia normal, i el R2 t'avisa que hi ha dies en què
s'equivoca de manera catastròfica.

Fixa't que aquí tenim més de dues mil cinc-centes dades, i tot i així el grau 4
sobreajusta. És la secció 7 vista des de l'altre costat: **dues mil cinc-centes
dades són moltes per a una corba de quinze coeficients, però poques per a un model
de 494.**

Resumint: la regressió múltiple sí que millora la predicció, però només quan el
model pot aprofitar el que porten les variables, i encara més quan les pot
combinar. El millor model d'aquesta secció, el de grau 3 (0,95 GWh), s'equivoca
menys de la meitat que la recta de la secció 3. A la **UT3** aprendràs a fabricar a
mà columnes com aquestes, triant les que tenen sentit en lloc de generar-les totes.

## 9. Ara sí: per què el test està segellat

Al NB 2.2 vam apartar un 20% dels districtes de Califòrnia i et vam prometre que
entendries per què al final d'aquest notebook. Avui hem fet el mateix amb el 2024 i
el 2025. I acabes de fer exactament el que ho explica.

A la secció 6 has entrenat quinze models, has mirat l'error de validació de
cadascun i t'has quedat amb el que en tenia menys. És el que s'ha de fer. Però
pensa què vol dir: **el grau 2 ha guanyat perquè era el que millor anava amb
aquestes dades concretes**. Amb uns altres quinze dies d'entrenament, el guanyador
podria haver estat el 3 o el 5 (ho has comprovat a l'exercici 3). L'1,5 GWh del
grau 2 és l'error del model que ha quedat primer en una competició, i al primer
lloc sempre hi ha una mica de sort. És un número lleugerament optimista. I el
mateix val per als 0,95 GWh de la secció 8: també és el guanyador d'una
competició, la dels quatre graus.

Una tria sola no fa gaire mal. El problema és que durant el curs en farem
centenars: aquesta variable sí, aquesta no, aquest grau, aquest model és millor
que aquell... Cada decisió que prenem mirant la validació **adapta una mica el
nostre treball a aquells dies concrets**. Al cap de tres mesos, el número de
validació ja no serà una estimació honesta de com funcionarà el model amb dies
nous. És una trampa que no es veu, perquè cap decisió sola és una trampa: és la
suma de totes.

Per això necessitem un tercer tros que **no hagi participat en cap decisió**, i
aquest és el test segellat. Recuperem la taula del NB 2.2, ara amb els anys
d'aquest fitxer i una columna nova:

| Tros | Per a què | Quantes vegades el mirem |
|---|---|---|
| **Entrenament** (2015-2021) | el model hi aprèn amb `fit()` | cada vegada que entrenem |
| **Validació** (2022-2023) | nosaltres hi comparem models i decidim | tantes com calgui |
| **Test segellat** (2024-2025) | comprovar, al final, si tot plegat funciona | **una sola vegada**, a la UT10 |

Si algun dia miréssim el test *només per curiositat* i després canviéssim alguna
cosa, encara que fos sense voler, el test passaria a ser una validació més i ja no
tindríem cap manera honesta de saber com funcionen els nostres models. Per això
els notebooks comencen sempre apartant el test segellat: no és desconfiança, és
l'única manera que el número final valgui alguna cosa.

## 10. Resum

- Una regressió lineal pot dibuixar corbes si li fabriquem columnes noves amb
  `PolynomialFeatures`. El **grau** és un hiperparàmetre que decideix quants
  revolts pot fer la corba.
- **Infraajust**: el model és massa simple i s'equivoca molt fins i tot amb les
  dades d'entrenament.
- **Sobreajust**: el model memoritza les dades d'entrenament i s'equivoca molt
  amb les noves.
- Es detecten comparant l'**error d'entrenament** amb l'**error de validació**.
  Una distància gran entre tots dos és el senyal d'alarma.
- L'error d'entrenament **no serveix per triar el model**: sempre prefereix el més
  complex.
- El sobreajust depèn de la complexitat del model **i** de la quantitat de dades.
- Amb el problema de veritat, les columnes polinòmiques milloren la predicció
  quan doblegen les variables (la corba de la temperatura) o les combinen. I amb
  massa columnes, també sobreajusten.
- Un MAE que gairebé no es mou pot amagar uns quants errors enormes. Per això
  convé mirar també el R2, que els castiga molt més.
- La validació serveix per decidir, i com que hi decidim tantes vegades, acaba
  sent una mica optimista. Per això el **test segellat** no participa en cap
  decisió i només s'obre una vegada, a la UT10.

## 11. Exercicis finals

Els tres exercicis *Ara tu* els has fet pel camí, cadascun lligat a una secció.
Aquests quatre últims et demanen fer servir el notebook sencer.

**4. Quaranta dies.** Repeteix la secció 6 amb 40 dies en lloc de 15
(`train.sample(40, random_state=1)`). On és ara el mínim de la validació? El
grau 15 és tan desastrós com abans?

**5. La demanda al llarg de l'any.** Repeteix la secció 4.1 canviant l'entrada:
en lloc de la temperatura, fes servir el moment de l'any, la fracció de l'any
(`dia_any / 366`). Dibuixa primer el núvol de punts i prova els graus 2, 4, 6 i 8.
Quin grau et sembla suficient? Per què en necessites més que amb la temperatura?
Amb el millor grau, l'error és més gran o més petit que amb la temperatura?
Pista: hauràs d'adaptar les funcions, que tenen `"temp_escalada"` escrit a dins.

**6. Per escrit.** Explica amb les teves paraules, en un paràgraf, la diferència
entre un model que *aprèn* i un model que *memoritza*. Fes servir com a exemple
alguna situació de fora de la informàtica que no sigui la de l'examen.

**7. Per escrit.** Un company proposa: *"Només per curiositat, calculem el MAE del
model sobre el 2024 i el 2025. No canviarem res, només mirarem."* Explica en unes
quatre línies per què no s'ha de fer, fent servir el que has après a la secció 9.

## 12. Per al debat de classe

- Un company us ensenya un model amb un R2 de 0,99 sobre les dades d'entrenament.
  Quina és la primera pregunta que li faríeu?
- Penseu en situacions reals on **només hi ha poques dades** i on, per tant, el
  risc de sobreajust és alt: un producte que fa un mes que es ven, una malaltia
  rara, una botiga que acaba d'obrir... Què faríeu en cada cas?
- Sempre és possible *aconseguir més dades*? Quins costos té?
- **La calor o els turistes?** A l'estiu fa calor i, alhora, a les illes hi ha
  molta més gent. Quina part de la pujada del consum és per l'aire condicionat i
  quina pels visitants? Amb les dades d'avui ho podríem saber? Quines dades hi
  afegiríeu?
- **El 2020.** Recordeu la franja de punts baixos de l'estiu. Si haguéssiu de
  predir la demanda del 2026, faríeu servir el 2020 per entrenar? Què hi guanyeu i
  què hi perdeu si el traieu?